# SSL Experiment — nb02: DINO-lite Baseline

**Goal:** Train a DINO-lite model (EMA teacher-student self-distillation) on CIFAR-10 and establish a second SSL baseline.

**Method:** Student learns to match the sharpened, centered output distribution of the teacher (EMA copy). Symmetric loss on both views.

**Outputs:**
- Linear probe accuracy (same protocol as nb01)
- KNN-20 accuracy
- UMAP side-by-side: student vs teacher embeddings
  (teacher expected to be smoother due to EMA averaging)

## 0. Setup

In [ ]:
import sys, os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_URL = 'https://github.com/jacobposchl/model-interpretability'
    REPO_DIR = '/content/ctls'

    if not os.path.exists(REPO_DIR):
        os.system(f"git clone {REPO_URL} {REPO_DIR}")
        os.system(f"pip install -r {REPO_DIR}/requirements.txt -q")

    from google.colab import drive
    drive.mount('/content/drive')

    DRIVE_BASE = '/content/drive/MyDrive/ctls'
    os.makedirs(f"{DRIVE_BASE}/data/raw", exist_ok=True)
    os.makedirs(f"{DRIVE_BASE}/experiments", exist_ok=True)

    for link, target in [
        (f"{REPO_DIR}/data/raw",    f"{DRIVE_BASE}/data/raw"),
        (f"{REPO_DIR}/experiments", f"{DRIVE_BASE}/experiments"),
    ]:
        if os.path.islink(link): os.unlink(link)
        os.symlink(target, link)

    ROOT = REPO_DIR
else:
    ROOT = os.path.abspath(os.path.join(os.getcwd(), '../..'))

if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.chdir(ROOT)
print(f"Working directory: {os.getcwd()} ({'Colab' if IN_COLAB else 'local'})")

In [ ]:
import yaml

with open('configs/ssl/dino.yaml') as f:
    config = yaml.safe_load(f)

print(yaml.dump(config, default_flow_style=False))

## 1. Train DINO-lite

Expected training time: ~40–55 min on GPU (200 epochs). Slightly slower than SimCLR due to the two-forward-pass teacher-student architecture.

**Skip if `experiments/ssl/dino/best.pt` already exists.**

In [ ]:
from training.ssl_trainer import SSLTrainer

trainer = SSLTrainer(config)
trainer.train()

## 2. Load Student and Teacher Models

In [ ]:
import torch
from models.soft_mask import SoftMask
from models.backbone import CTLSBackbone
from models.meta_encoder import MetaEncoder
from models.momentum_encoder import MomentumEncoder

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

mcfg = config['model']
ecfg = mcfg['meta_encoder']

# Student
soft_mask = SoftMask(init_temperature=mcfg.get('soft_mask', {}).get('init_temperature', 1.0))
backbone = CTLSBackbone(
    arch=mcfg['arch'], num_classes=mcfg['num_classes'],
    soft_mask=soft_mask, pretrained=False,
).to(DEVICE)
meta_encoder = MetaEncoder(
    layer_dims=backbone.layer_dims,
    hidden_dim=ecfg.get('hidden_dim', 256),
    embedding_dim=ecfg.get('embedding_dim', 64),
    encoder_type=ecfg.get('encoder_type', 'mlp'),
).to(DEVICE)

ckpt = torch.load('experiments/ssl/dino/best.pt', map_location=DEVICE)
backbone.load_state_dict(ckpt['backbone_state'])
meta_encoder.load_state_dict(ckpt['meta_encoder_state'])
backbone.eval(); meta_encoder.eval()

# Teacher (from checkpoint)
teacher = MomentumEncoder(backbone, meta_encoder, momentum=0.996).to(DEVICE)
teacher.load_state_dict(ckpt['teacher_state'])
teacher.eval()

print(f"Loaded DINO checkpoint (epoch {ckpt['epoch']})")

## 3. Linear Probe and KNN-20

In [ ]:
from data.cifar import get_standard_loaders
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import TensorDataset, DataLoader as DL
from sklearn.neighbors import KNeighborsClassifier

dcfg = config['data']
train_loader_labeled, val_loader = get_standard_loaders(
    data_dir=dcfg['data_dir'], batch_size=256, num_workers=4, augment=True, download=True,
)

def collect_embeddings(backbone, meta_encoder, loader, device):
    all_z, all_y = [], []
    with torch.no_grad():
        for x, y in loader:
            _, traj = backbone(x.to(device))
            z = meta_encoder(traj)
            all_z.append(z.cpu()); all_y.append(y)
    return torch.cat(all_z), torch.cat(all_y)

z_train, y_train = collect_embeddings(backbone, meta_encoder, train_loader_labeled, DEVICE)
z_val, y_val = collect_embeddings(backbone, meta_encoder, val_loader, DEVICE)

# Linear probe
EMBEDDING_DIM = ecfg.get('embedding_dim', 64)
linear_probe = nn.Linear(EMBEDDING_DIM, 10).to(DEVICE)
probe_opt = AdamW(linear_probe.parameters(), lr=1e-3, weight_decay=1e-4)
probe_ds = DL(TensorDataset(z_train, y_train), batch_size=256, shuffle=True)

for ep in range(100):
    linear_probe.train()
    for z, y in probe_ds:
        loss = F.cross_entropy(linear_probe(z.to(DEVICE)), y.to(DEVICE))
        probe_opt.zero_grad(); loss.backward(); probe_opt.step()

linear_probe.eval()
with torch.no_grad():
    preds = linear_probe(z_val.to(DEVICE)).argmax(dim=-1).cpu()
lp_acc = (preds == y_val).float().mean().item()

# KNN-20
knn = KNeighborsClassifier(n_neighbors=20, metric='cosine')
knn.fit(z_train.numpy(), y_train.numpy())
knn_acc = (knn.predict(z_val.numpy()) == y_val.numpy()).mean()

print(f"Linear probe accuracy (DINO): {lp_acc:.4f}")
print(f"KNN-20 accuracy      (DINO): {knn_acc:.4f}")

## 4. UMAP: Student vs Teacher

The teacher (EMA copy) should show slightly smoother, more stable cluster boundaries
compared to the student because it is an average over all student checkpoints.

In [ ]:
from evaluation.circuit_viz import CircuitVisualizer
import matplotlib.pyplot as plt

# Teacher uses its own backbone/meta_encoder (the EMA copies)
teacher_backbone = teacher.teacher_backbone
teacher_meta_enc = teacher.teacher_meta_encoder

viz_student = CircuitVisualizer(backbone, meta_encoder, val_loader, DEVICE)
viz_teacher = CircuitVisualizer(teacher_backbone, teacher_meta_enc, val_loader, DEVICE)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

print('Computing student UMAP...')
viz_student.plot_umap(title='DINO Student', compare_output=False,
                      max_samples=5000, ax=axes[0] if hasattr(viz_student, '_plot_to_ax') else None)

# Standalone plots if ax injection not supported
fig_s = viz_student.plot_umap(title='DINO Student — circuit space', compare_output=False, max_samples=5000)
fig_t = viz_teacher.plot_umap(title='DINO Teacher (EMA) — circuit space', compare_output=False, max_samples=5000)

os.makedirs('experiments/ssl/dino', exist_ok=True)
fig_s.savefig('experiments/ssl/dino/umap_student.png', dpi=150, bbox_inches='tight')
fig_t.savefig('experiments/ssl/dino/umap_teacher.png', dpi=150, bbox_inches='tight')
plt.show()

s_scores = viz_student.cluster_separation_score(max_samples=5000)
t_scores = viz_teacher.cluster_separation_score(max_samples=5000)
print(f"\nStudent circuit silhouette: {s_scores['silhouette_circuit']:.4f}")
print(f"Teacher circuit silhouette: {t_scores['silhouette_circuit']:.4f}")

## Summary

| Metric | SimCLR (nb01) | DINO-lite |
|--------|--------------|----------|
| Linear probe accuracy | ___ | ___ |
| KNN-20 accuracy | ___ | ___ |
| Circuit silhouette | ___ | ___ |

**Next:** Run `nb03_ctls_ssl_training.ipynb`.